# Proyecto Grupal — Entrenar y exportar el modelo para el predictor (Gradio)

Reentrena exactamente el modelo final de la Práctica 2 (`modelo_combinado`: embeddings para
`producto`/`subcategoria`, one-hot para el resto, variables de historial de cliente/tienda),
y **exporta todo lo necesario** para que la app de Gradio prediga sin tener que reentrenar
nada: el modelo, el preprocesador, los vocabularios de embeddings, y tablas de valores
típicos (precio por producto, categoría por producto, etc.) para completar solas las
variables que el usuario no va a ingresar en el formulario simple.

**Al final de este notebook vas a tener una carpeta `/content/artefactos_predictor` con
todos los archivos para subir al repositorio de Hugging Face Spaces**, junto a `app.py`.


## 1. Datos y features (idéntico a Práctica 2)

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os, random, glob, json, joblib

drive.mount('/content/drive')
candidatos = glob.glob('/content/drive/MyDrive/**/ventas_adventureworks*.csv', recursive=True)
file_path = candidatos[0]
df = pd.read_csv(file_path)

SEMILLA = 42
os.environ["PYTHONHASHSEED"] = str(SEMILLA)
random.seed(SEMILLA)
np.random.seed(SEMILLA)
tf.random.set_seed(SEMILLA)
print("Filas:", df.shape[0])


In [ ]:
df_procesado = df.copy()
for columna in ["orderdate", "iniciooferta", "finoferta"]:
    df_procesado[columna] = pd.to_datetime(df_procesado[columna], errors="coerce")

df_procesado["anio"] = df_procesado["orderdate"].dt.year
df_procesado["mes"] = df_procesado["orderdate"].dt.month
df_procesado["dia"] = df_procesado["orderdate"].dt.day
df_procesado["diasemana"] = df_procesado["orderdate"].dt.dayofweek
df_procesado["semanaanio"] = df_procesado["orderdate"].dt.isocalendar().week.astype(int)
df_procesado["canalventa"] = np.select(
    [df_procesado["onlineorderflag"].eq(1), df_procesado["storeid"].notna()],
    ["Internet", "Tienda"], default="Cliente individual"
)
df_procesado["tienemaxqty"] = df_procesado["maxqty"].notna().astype(int)
df_procesado["maxqty"] = df_procesado["maxqty"].fillna(0)
df_procesado["duracionoferta"] = (df_procesado["finoferta"] - df_procesado["iniciooferta"]).dt.days
df_procesado["diasdesdeiniciooferta"] = (df_procesado["orderdate"] - df_procesado["iniciooferta"]).dt.days

conteo_diario = df_procesado.groupby("orderdate").size().sort_index()
acumulado = conteo_diario.cumsum()
total = conteo_diario.sum()
fecha_corte_train = acumulado[acumulado >= total * 0.70].index[0]
fecha_corte_validation = acumulado[acumulado >= total * 0.85].index[0]

train = df_procesado[df_procesado["orderdate"] < fecha_corte_train].copy()
validation = df_procesado[(df_procesado["orderdate"] >= fecha_corte_train) & (df_procesado["orderdate"] < fecha_corte_validation)].copy()
test = df_procesado[df_procesado["orderdate"] >= fecha_corte_validation].copy()
print("Train:", len(train), "Validación:", len(validation), "Prueba:", len(test))


In [ ]:
promedio_por_cliente = train.groupby("customerid")["cantidadvendida"].mean()
promedio_global_cliente = train["cantidadvendida"].mean()
promedio_por_tienda = train.groupby("storeid")["cantidadvendida"].mean()
promedio_global_tienda = train["cantidadvendida"].mean()
conteo_por_cliente = train.groupby("customerid").size()
conteo_global_cliente = conteo_por_cliente.median()

for conjunto in [train, validation, test]:
    conjunto["cliente_prom_hist"] = conjunto["customerid"].map(promedio_por_cliente).fillna(promedio_global_cliente)
    conjunto["cliente_frecuencia_hist"] = conjunto["customerid"].map(conteo_por_cliente).fillna(conteo_global_cliente)
    conjunto["tienda_prom_hist"] = conjunto["storeid"].map(promedio_por_tienda).fillna(promedio_global_tienda)

train["cliente_nuevo"] = 0
validation["cliente_nuevo"] = (~validation["customerid"].isin(promedio_por_cliente.index)).astype(int)
test["cliente_nuevo"] = (~test["customerid"].isin(promedio_por_cliente.index)).astype(int)


## 2. Preparación de variables (igual que el modelo combinado de P2)

In [ ]:
target = "cantidadvendida"
columnas_no_predictoras = [
    target, "orderdate", "iniciooferta", "finoferta", "productid", "specialofferid",
    "salesorderid", "salesorderdetailid", "purchaseordernumber", "customerid", "storeid",
    "onlineorderflag", "linetotal", "color", "style", "size"
]

X_train = train.drop(columns=columnas_no_predictoras, errors="ignore")
X_validation = validation.drop(columns=columnas_no_predictoras, errors="ignore")
X_test = test.drop(columns=columnas_no_predictoras, errors="ignore")
y_train = train[target].astype("float32")
y_validation = validation[target].astype("float32")
y_test = test[target].astype("float32")

columnas_numericas = X_train.select_dtypes(include=["number"]).columns.tolist()
columnas_categoricas = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

cols_embedding = ["producto", "subcategoria"]
columnas_categoricas_resto = [c for c in columnas_categoricas if c not in cols_embedding]

vocab_emb = {}
emb_dim = {}
for col in cols_embedding:
    categorias = sorted(X_train[col].astype(str).unique())
    vocab_emb[col] = {v: i + 1 for i, v in enumerate(categorias)}
    cardinalidad = len(categorias) + 1
    emb_dim[col] = int(max(2, min(50, (cardinalidad + 1) // 2)))

def codificar(df_split, col):
    return df_split[col].astype(str).map(lambda v: vocab_emb[col].get(v, 0)).astype("int32").values

preprocesador_resto = ColumnTransformer(transformers=[
    ("numericas", StandardScaler(), columnas_numericas),
    ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_categoricas_resto)
])
X_train_resto = preprocesador_resto.fit_transform(X_train).astype("float32")
X_validation_resto = preprocesador_resto.transform(X_validation).astype("float32")
X_test_resto = preprocesador_resto.transform(X_test).astype("float32")

def armar_inputs(df_split, X_resto):
    d = {col: codificar(df_split, col) for col in cols_embedding}
    d["resto"] = X_resto
    return d

Xtr_comb = armar_inputs(X_train, X_train_resto)
Xva_comb = armar_inputs(X_validation, X_validation_resto)
Xte_comb = armar_inputs(X_test, X_test_resto)
print("Dimensión 'resto':", X_train_resto.shape)


## 3. Entrenar el modelo combinado (arquitectura idéntica a la de Práctica 2)

In [ ]:
def crear_modelo_combinado(semilla=42):
    tf.keras.backend.clear_session()
    keras.utils.set_random_seed(semilla)
    entradas, embeddings = [], []
    for col in cols_embedding:
        entrada = keras.Input(shape=(1,), name=col, dtype="int32")
        entradas.append(entrada)
        emb = layers.Embedding(len(vocab_emb[col]) + 1, emb_dim[col], name=f"emb_{col}")(entrada)
        embeddings.append(layers.Flatten()(emb))
    entrada_resto = keras.Input(shape=(X_train_resto.shape[1],), name="resto")
    entradas.append(entrada_resto)
    x = layers.Concatenate()(embeddings + [entrada_resto])
    x = layers.Dense(128)(x); x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x); x = layers.Dropout(0.20)(x)
    x = layers.Dense(64)(x); x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x); x = layers.Dropout(0.20)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dense(16, activation="relu")(x)
    salida = layers.Dense(1)(x)
    modelo = keras.Model(entradas, salida)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss="mse",
                    metrics=[keras.metrics.MeanAbsoluteError(name="mae"), keras.metrics.RootMeanSquaredError(name="rmse")])
    return modelo

modelo_combinado = crear_modelo_combinado()
early_stopping_comb = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
historial_comb = modelo_combinado.fit(
    Xtr_comb, y_train, validation_data=(Xva_comb, y_validation),
    epochs=100, batch_size=256, callbacks=[early_stopping_comb], verbose=1
)


In [ ]:
def evaluar_modelo(modelo, X, y):
    pred = modelo.predict(X, verbose=0).flatten()
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = r2_score(y, pred)
    return mae, rmse, r2, pred

mae_t, rmse_t, r2_t, pred_t = evaluar_modelo(modelo_combinado, Xte_comb, y_test)
print(f"Prueba -> MAE {mae_t:.4f}  RMSE {rmse_t:.4f}  R2 {r2_t:.4f}")
print("(Referencia esperada, del informe de P2: MAE 0.4282 RMSE 1.0223 R2 0.7440 — puede variar levemente entre corridas.)")


## 4. Exportar todo lo necesario para el predictor

Se calculan tablas de valores típicos (precio por producto, categoría por producto, etc.)
para que el formulario simple de Gradio (solo producto, canal de venta, y si hay oferta)
pueda completar el resto de las variables de forma razonable, sin pedírselas al usuario.

In [ ]:
os.makedirs("/content/artefactos_predictor", exist_ok=True)

# 1) Modelo entrenado
modelo_combinado.save("/content/artefactos_predictor/modelo_combinado.keras")

# 2) Preprocesador (ColumnTransformer ya ajustado con train)
joblib.dump(preprocesador_resto, "/content/artefactos_predictor/preprocesador_resto.joblib")

# 3) Vocabularios y dimensiones de los embeddings
with open("/content/artefactos_predictor/vocab_emb.json", "w", encoding="utf-8") as f:
    json.dump(vocab_emb, f, ensure_ascii=False)
with open("/content/artefactos_predictor/emb_dim.json", "w", encoding="utf-8") as f:
    json.dump(emb_dim, f)

# 4) Listas de columnas (para reconstruir el DataFrame de una fila en el mismo orden)
with open("/content/artefactos_predictor/columnas.json", "w", encoding="utf-8") as f:
    json.dump({
        "numericas": columnas_numericas,
        "categoricas_resto": columnas_categoricas_resto,
        "cols_embedding": cols_embedding,
    }, f, ensure_ascii=False)

# 5) Jerarquía producto -> subcategoría -> categoría (determinística en AdventureWorks)
jerarquia = (
    train[["producto", "subcategoria", "categoria"]]
    .drop_duplicates(subset=["producto"])
    .set_index("producto")[["subcategoria", "categoria"]]
    .to_dict(orient="index")
)
with open("/content/artefactos_predictor/jerarquia_producto.json", "w", encoding="utf-8") as f:
    json.dump(jerarquia, f, ensure_ascii=False)

# 6) Precio típico (mediana) por producto
precios_por_producto = train.groupby("producto")[["unitprice", "listprice", "standardcost"]].median().to_dict(orient="index")
with open("/content/artefactos_predictor/precios_por_producto.json", "w", encoding="utf-8") as f:
    json.dump(precios_por_producto, f, ensure_ascii=False)

# 7) Valores por defecto, separados según si la venta "simulada" tiene oferta o no
columnas_oferta_dependientes = ["unitpricediscount", "discountpct", "minqty", "maxqty", "tienemaxqty", "duracionoferta", "diasdesdeiniciooferta"]
con_oferta = train[train["oferta"] != "No Discount"]
sin_oferta = train[train["oferta"] == "No Discount"]

defaults_sin_oferta = sin_oferta[columnas_oferta_dependientes].median(numeric_only=True).to_dict()
defaults_sin_oferta["oferta"] = "No Discount"
defaults_sin_oferta["categoriaoferta"] = sin_oferta["categoriaoferta"].mode().iloc[0]

defaults_con_oferta = con_oferta[columnas_oferta_dependientes].median(numeric_only=True).to_dict()
defaults_con_oferta["oferta"] = con_oferta["oferta"].mode().iloc[0]
defaults_con_oferta["categoriaoferta"] = con_oferta["categoriaoferta"].mode().iloc[0]

# 8) Otros valores globales (territorio, historial "cliente promedio")
defaults_globales = {
    "territoryid": float(train["territoryid"].mode().iloc[0]),
    "cliente_prom_hist": float(promedio_global_cliente),
    "cliente_frecuencia_hist": float(conteo_global_cliente),
    "tienda_prom_hist": float(promedio_global_tienda),
    "cliente_nuevo": 0,
    "categoria_default": train["categoria"].mode().iloc[0],
}

with open("/content/artefactos_predictor/defaults.json", "w", encoding="utf-8") as f:
    json.dump({
        "sin_oferta": defaults_sin_oferta,
        "con_oferta": defaults_con_oferta,
        "globales": defaults_globales,
    }, f, ensure_ascii=False)

# 9) Listas para los desplegables del formulario
opciones = {
    "productos": sorted(train["producto"].astype(str).unique().tolist()),
    "canales": sorted(train["canalventa"].astype(str).unique().tolist()),
}
with open("/content/artefactos_predictor/opciones.json", "w", encoding="utf-8") as f:
    json.dump(opciones, f, ensure_ascii=False)

print("Listo. Archivos generados en /content/artefactos_predictor:")
for archivo in sorted(os.listdir("/content/artefactos_predictor")):
    print(" -", archivo)


In [ ]:
# Comprimir todo en un .zip para descargar fácilmente
import shutil
shutil.make_archive("/content/artefactos_predictor", "zip", "/content/artefactos_predictor")
print("Descargá /content/artefactos_predictor.zip desde el panel de archivos de Colab (ícono de carpeta a la izquierda).")


## 5. Próximo paso

Descargá `artefactos_predictor.zip`, descomprimilo, y subí **todos los archivos de adentro**
(no la carpeta, los archivos sueltos) al repositorio de tu Hugging Face Space, junto con
`app.py` y `requirements.txt` (ver notebook/archivos aparte para eso).


## 6. Predictor con Gradio — link público directo desde Colab

No hace falta desplegar nada en Hugging Face: Gradio genera un **link público temporal**
(`https://xxxxx.gradio.live`, válido ~72 horas) directo desde esta misma sesión de Colab.
Como ya tenemos `modelo_combinado`, `preprocesador_resto`, `jerarquia`, etc. en memoria (de
las celdas anteriores), no hace falta volver a cargar ningún archivo.


In [ ]:
!pip install -q gradio
import gradio as gr
from datetime import date


In [ ]:
def _codificar_embedding(valor, col):
    return np.array([vocab_emb[col].get(str(valor), 0)], dtype="int32")


def construir_fila(producto, canal_venta, hay_oferta):
    hoy = date.today()
    info_producto = jerarquia.get(producto, {})
    subcategoria = info_producto.get("subcategoria", defaults_globales["categoria_default"])
    categoria = info_producto.get("categoria", defaults_globales["categoria_default"])

    precios = precios_por_producto.get(
        producto, {"unitprice": 500.0, "listprice": 550.0, "standardcost": 300.0}
    )
    grupo = defaults_con_oferta if hay_oferta else defaults_sin_oferta

    fila = {
        "territoryid": defaults_globales["territoryid"],
        "unitprice": precios["unitprice"],
        "unitpricediscount": grupo["unitpricediscount"],
        "listprice": precios["listprice"],
        "standardcost": precios["standardcost"],
        "discountpct": grupo["discountpct"],
        "minqty": grupo["minqty"],
        "maxqty": grupo["maxqty"],
        "anio": hoy.year,
        "mes": hoy.month,
        "dia": hoy.day,
        "diasemana": hoy.weekday(),
        "semanaanio": int(hoy.isocalendar()[1]),
        "tienemaxqty": grupo["tienemaxqty"],
        "duracionoferta": grupo["duracionoferta"],
        "diasdesdeiniciooferta": grupo["diasdesdeiniciooferta"],
        "cliente_prom_hist": defaults_globales["cliente_prom_hist"],
        "cliente_frecuencia_hist": defaults_globales["cliente_frecuencia_hist"],
        "tienda_prom_hist": defaults_globales["tienda_prom_hist"],
        "cliente_nuevo": defaults_globales["cliente_nuevo"],
        "categoria": categoria,
        "oferta": grupo["oferta"],
        "categoriaoferta": grupo["categoriaoferta"],
        "canalventa": canal_venta,
    }
    orden = columnas_numericas + columnas_categoricas_resto
    return pd.DataFrame([fila])[orden], subcategoria


def predecir(producto, canal_venta, hay_oferta):
    if not producto or not canal_venta:
        return "Elegí un producto y un canal de venta para predecir."

    fila_df, subcategoria = construir_fila(producto, canal_venta, hay_oferta)
    X_resto = preprocesador_resto.transform(fila_df).astype("float32")
    entradas = {
        "producto": _codificar_embedding(producto, "producto"),
        "subcategoria": _codificar_embedding(subcategoria, "subcategoria"),
        "resto": X_resto,
    }
    prediccion = max(0.0, float(modelo_combinado.predict(entradas, verbose=0).flatten()[0]))

    if prediccion < 1.5:
        comentario = "Pedido típico (1 unidad) — es el caso más común en los datos históricos."
    elif prediccion < 4:
        comentario = "Pedido moderado."
    else:
        comentario = "Pedido grande — poco frecuente en los datos históricos, el modelo tiende a subestimar estos casos."

    return (
        f"### Cantidad estimada: **{prediccion:.1f} unidades**\n\n"
        f"*Producto:* {producto}  (subcategoría: {subcategoria})\n\n"
        f"*Canal:* {canal_venta} · *Oferta activa:* {'Sí' if hay_oferta else 'No'}\n\n"
        f"{comentario}"
    )


In [ ]:
productos_disponibles = sorted(train["producto"].astype(str).unique().tolist())
canales_disponibles = sorted(train["canalventa"].astype(str).unique().tolist())

descripcion = """
## Predictor de cantidad de venta — AdventureWorks
Estimá cuántas unidades se venderían en una línea de pedido, según el producto,
el canal de venta, y si hay una oferta activa. Modelo: DNN con *entity embeddings*,
entrenado sobre el historial real de ventas (Proyecto Grupal 12).
"""

with gr.Blocks(title="Predictor de ventas — AdventureWorks") as demo:
    gr.Markdown(descripcion)
    with gr.Row():
        with gr.Column():
            producto_input = gr.Dropdown(choices=productos_disponibles, label="Producto", value=productos_disponibles[0])
            canal_input = gr.Dropdown(choices=canales_disponibles, label="Canal de venta", value=canales_disponibles[0])
            oferta_input = gr.Checkbox(label="¿Hay una oferta/descuento activo?", value=False)
            boton = gr.Button("Predecir cantidad", variant="primary")
        with gr.Column():
            salida = gr.Markdown(label="Resultado")
    boton.click(fn=predecir, inputs=[producto_input, canal_input, oferta_input], outputs=salida)

demo.launch(share=True)  # genera el link público https://xxxxx.gradio.live


**Al correr la última celda:** Colab va a tardar unos segundos y te va a imprimir dos links —
uno local (`http://127.0.0.1:...`, no sirve fuera de Colab) y uno público
(`https://xxxxx.gradio.live`, **ese es el que usás para grabar el video** y compartir con el
docente). Dejá la celda corriendo mientras grabás — si la interrumpís o se desconecta Colab,
el link se cae.
